# Begleit-Notebook zu Kapitel 17
**Build Your First LLM — Kapitel 17: Deployment-Optionen**

Die Zellen geben Vorlagen und Terminalbefehle aus. Das Notebook benötigt weder Zugangsdaten noch `llm_helper.py`: Es installiert kein Modal, sendet keine Requests, startet keine GPU und stellt nichts bereit. Nur wenn du selbst den gehosteten Weg wählst, führst du `modal setup` und die Befehle in deinem lokalen Terminal mit deinem Modal-Konto aus. Dabei können Kosten entstehen.

In [ ]:
# Dieses Notebook gibt Quellcode aus. Installiere die Pakete nur für ein echtes Deployment.
print("Optional im lokalen Terminal (virtuelle Umgebung, Python 3.11):")
print("python -m pip install modal fastapi pydantic requests")
print("modal setup  # CLI-Anmeldung, kein Proxy-Token")

## Abschnitt 17.1: Warum Modal?

**Was ist Serverless?**

| Traditioneller Server | Serverless (Modal) |
|-------------------|-------------------|
| Du mietest einen Computer 24/7 | Code läuft nur bei Bedarf |
| Du zahlst auch im Leerlauf | Du zahlst pro Sekunde Rechenzeit |
| Du verwaltest Updates, Sicherheit | Die Plattform verwaltet die Infrastruktur; du schützt den App-Zugriff |
| Feste Kapazität | Automatische Skalierung |

**Analogie:** Traditionelles Hosting ist wie ein eigenes Auto zu besitzen. Serverless ist wie ein Taxi zu nutzen – du zahlst nur, wenn du fährst.

**Warum Modal für LLMs?**
- GPU-Unterstützung eingebaut (T4 bis H100)
- 30$/Monat kostenlose Credits (~37 Stunden L4-GPU-Zeit; ~50 Stunden auf T4)
- Python-nativ (kein Docker, kein YAML)
- Skaliert auf Null (keine Gebühren im Leerlauf)

## Abschnitt 17.2: Hallo Modal

Die einfachste Modal-App. Speichere dies als `hello.py` und führe es mit `modal run hello.py` aus:

In [ ]:
# Speichere dies als hello.py
hello_modal_code = '''
import modal

app = modal.App("hello-world")

@app.function()
def hello(name: str) -> str:
    return f"Hallo, {name}!"

@app.local_entrypoint()
def main():
    result = hello.remote("Welt")
    print(result)
'''

print("Speichere diesen Code als 'hello.py':")
print(hello_modal_code)
print("\nDann ausführen: modal run hello.py")

**Was ist gerade passiert?**

1. `modal.App()` erstellt deine Anwendung
2. `@app.function()` markiert Code, der in Modals Cloud ausgeführt werden soll
3. `hello.remote()` ruft die Funktion remote auf
4. Erst wenn du den Befehl im Terminal ausführst, startet Modal einen Container, führt den Code aus und gibt das Ergebnis zurück

## Abschnitt 17.3: GPU-Unterstützung hinzufügen

Ein Parameter fügt GPU-Unterstützung hinzu:

In [ ]:
# Speichere dies als gpu_test.py
gpu_test_code = '''
import modal

app = modal.App("gpu-test")

# torch muss ins Container-Image installiert werden (nächster Abschnitt: Images)
image = modal.Image.debian_slim(python_version="3.11").pip_install("torch")

@app.function(image=image, gpu="L4")  # Das war's!
def check_gpu():
    import torch
    if torch.cuda.is_available():
        device = torch.cuda.get_device_name(0)
        return f"GPU verfügbar: {device}"
    return "Keine GPU gefunden"

@app.local_entrypoint()
def main():
    print(check_gpu.remote())
'''

print("Speichere diesen Code als 'gpu_test.py':")
print(gpu_test_code)
print("\nDann ausführen: modal run gpu_test.py")
print("\nErwartete Ausgabe: GPU verfügbar: NVIDIA L4")

**Ein Parameter.** Keine CUDA-Installation, keine Treiberverwaltung, keine Docker-Images. Modal übernimmt den gesamten GPU-Stack.

## Abschnitt 17.4: Abhängigkeiten installieren

Dein LLM benötigt Bibliotheken. Modal ermöglicht es dir, deine Container-Umgebung in Python zu definieren:

In [ ]:
# Speichere dies als with_deps.py
with_deps_code = '''
import modal

# Container-Image definieren
image = modal.Image.debian_slim(python_version="3.11").pip_install(
    "torch",
    "transformers",
    "accelerate",
)

app = modal.App("with-deps")

@app.function(image=image, gpu="T4")
def generate_text():
    from transformers import pipeline
    
    generator = pipeline("text-generation", model="gpt2", device=0)
    result = generator("Der Sinn des Lebens ist", max_length=50)
    return result[0]["generated_text"]

@app.local_entrypoint()
def main():
    print(generate_text.remote())
'''

print("Speichere diesen Code als 'with_deps.py':")
print(with_deps_code)
print("\nDann ausführen: modal run with_deps.py")

Der `image`-Parameter teilt Modal mit, was installiert werden soll. Der erste Durchlauf dauert länger (Image wird erstellt), aber nachfolgende Durchläufe verwenden das gecachte Image wieder.

## Abschnitt 17.5: Web-Endpunkte mit FastAPI

Erstelle einen Web-Endpunkt, der nur Anfragen mit gültigen Proxy-Zugangsdaten annimmt. Hier zahlt sich das FastAPI-Wissen aus Kapitel 16 aus:

Erstelle unter **Settings → Proxy auth tokens** oder mit `modal workspace proxy-tokens create --name chapter17` ein Proxy-Token. Verwahre die `wk-…`-ID und das nur einmal angezeigte `ws-…`-Geheimnis im Passwortmanager. Nutze eine separate Demo-Umgebung und beschränke das Token per RBAC darauf, falls verfügbar. Sein Name bindet es nicht an eine einzelne URL. Das Token gehört weder ins Notebook noch in ein Browser-Frontend.

In [ ]:
# Speichere dies als simple_api.py
simple_api_code = '''
import modal
from fastapi import FastAPI

app = modal.App("my-api")
web_app = FastAPI()

@web_app.get("/health")
def health():
    return {"status": "healthy"}

@web_app.get("/hello/{name}")
def hello(name: str):
    return {"message": f"Hallo, {name}!"}

image = modal.Image.debian_slim(python_version="3.11").pip_install(
    "fastapi==0.163.2", "uvicorn==0.54.0", "httpx==0.28.1", "pydantic==2.14.3"
)

@app.function(image=image)
@modal.asgi_app(requires_proxy_auth=True)
def serve():
    return web_app
'''

print("Speichere diesen Code als 'simple_api.py':")
print(simple_api_code)
print("\nDann ausführen: modal deploy simple_api.py")
print("\nDu erhältst eine URL wie: https://your-workspace--my-api-serve.modal.run")

Setze `SIMPLE_URL` in einer privaten Bash-Sitzung auf die HTTPS-Basis-URL deines eigenen Deployments. Lade `MODAL_KEY` und `MODAL_SECRET` aus dem Secret-Manager; tippe keine Geheimnisse in die Shell-Historie und deaktiviere Shell-Tracing.

```bash
curl -i --max-time 30 "$SIMPLE_URL/health"  # Ohne Header: HTTP 401 erwartet
curl -i --max-time 30 --header @- "$SIMPLE_URL/health" <<EOF
Modal-Key: $MODAL_KEY
Modal-Secret: $MODAL_SECRET
EOF
modal app stop my-api
```

Mit gültigen Headern erwartet: HTTP 200 und `{"status":"healthy"}`. Das prüft kein Modell. Auch `/hello/Welt` und die Dokumentation verlangen beide Header. Eine URL allein berechtigt nicht zum Zugriff. Auch bei einem Fehler: Stoppe `my-api` und kontrolliere im Dashboard, dass keine Container mehr laufen.

## Abschnitt 17.6: Der vollständige LLM-Service

Diese Zelle gibt die Service-Vorlage aus. Sie prüft weder den Modal-Proxy noch den GPU-Pfad.

In [ ]:
# Speichere dies als llm_service.py
# (Derselbe Service wie im Buch: Guardrails aus Kapitel 16 + vllm-serve-Subprozess.
#  API-verifiziert anhand der vLLM/Modal-Dokumentation 2026-11; Deployment braucht ein Modal-Konto + GPU.)
llm_service_code = '''
import modal
from fastapi import FastAPI, HTTPException, Request, Depends
from fastapi.security import HTTPBearer, HTTPAuthorizationCredentials
from fastapi.responses import JSONResponse
from pydantic import BaseModel, field_validator
import os
from collections import defaultdict
import threading
import uuid
from datetime import datetime
import subprocess
import time
import requests

MODEL = "Qwen/Qwen3.5-2B"
# Hinweis: kein trust_remote_code nötig. Aktuelle Modelle wie Qwen3.5
# werden von vLLM nativ unterstützt; aktiviere es nur für Repos, deren
# Custom-Code du geprüft hast, denn es führt Code aus dem Modell-Repo aus.

image = modal.Image.debian_slim(python_version="3.11").pip_install(
    "fastapi",
    "vllm==0.34.0",  # Pin für Reproduzierbarkeit (aktuelles Stable, geprüft 2026-11)
    "torch",
    "transformers",
    "requests",
)

app = modal.App("my-llm-service")
model_cache = modal.Volume.from_name("model-cache", create_if_missing=True)
web_app = FastAPI(title="My LLM API", version="1.0.0")

# Das Bearer-Schema zu deklarieren liefert Swaggers "Authorize"-Button und markiert
# geschützte Routen in den Docs; require_token unten macht die eigentliche Prüfung.
bearer_scheme = HTTPBearer(auto_error=False)

# ===== Leitplanken aus Kapitel 16, plus Auth und Token-Budgets =====
MAX_PROMPT_CHARS = 4000  # billiger erster Filter; das echte Budget ist in Tokens
API_TOKEN = os.environ.get("API_TOKEN", "")  # via Modal Secret gesetzt, siehe unten
MAX_INPUT_TOKENS = 2048  # Budget für den getemplateten Input (mit dem Output-Cap passt es in 8192)
MAX_OUTPUT_TOKENS = 512  # serverseitiger Cap: kein Client kann dir mehr in Rechnung stellen
RATE_LIMIT = 20
RATE_WINDOW = 60  # Sekunden auf der monotonen Uhr
request_counts: dict[str, list[float]] = defaultdict(list)
_rate_lock = threading.Lock()

def require_token(credentials: HTTPAuthorizationCredentials | None = Depends(bearer_scheme)):
    """Bearer-Token-Auth: ein geteilter GPU-Endpoint darf nicht der ganzen Welt offenstehen."""
    if not API_TOKEN or credentials is None or credentials.credentials != API_TOKEN:
        raise HTTPException(401, "Missing or invalid token")

class ChatRequest(BaseModel):
    message: str
    max_tokens: int = 256

    @field_validator("message")
    @classmethod
    def not_empty(cls, v):
        if not v.strip():
            raise ValueError("Message cannot be empty")
        if len(v) > MAX_PROMPT_CHARS:
            raise ValueError(f"Message too long (max {MAX_PROMPT_CHARS} chars)")
        return v

    @field_validator("max_tokens")
    @classmethod
    def cap_output(cls, v):
        if v < 1:
            raise ValueError("max_tokens must be positive")
        # Begrenzen statt ablehnen: Clients, die mehr wollen, bekommen den Cap, keinen Fehler
        return min(v, MAX_OUTPUT_TOKENS)

class ChatResponse(BaseModel):
    response: str
    latency_ms: float

def check_rate_limit(request: Request):
    client_ip = request.client.host
    now = time.monotonic()
    # Prüfen und Aktualisieren unter einem Lock: diese Endpoints laufen in
    # FastAPIs Threadpool, ohne den Lock könnten zwei gleichzeitige Requests
    # beide len == RATE_LIMIT - 1 lesen und beide durchschlüpfen.
    with _rate_lock:
        recent = [t for t in request_counts[client_ip] if now - t < RATE_WINDOW]
        if len(recent) >= RATE_LIMIT:
            request_counts[client_ip] = recent
            raise HTTPException(429, "Too many requests. Please wait a minute.")
        recent.append(now)
        request_counts[client_ip] = recent

def log_request(request_id, input_tokens, output_tokens, latency_ms, status):
    # Nur Metadaten von der Allowlist: niemals Prompts, Antworten oder rohen
    # Exception-Text loggen. Modal erfasst stdout.
    import json
    print(json.dumps({"ts": datetime.now().isoformat(),
                      "request_id": request_id,
                      "input_tokens": input_tokens,
                      "output_tokens": output_tokens,
                      "latency_ms": round(latency_ms, 2),
                      "status": status}))

# ===== vLLM-Server-Verwaltung (einer pro Container) =====
_server = None
_tokenizer = None

def start_vllm():
    """vllm serve einmal pro Container starten und warten, bis er bereit ist."""
    global _server, _tokenizer
    if _server is None:
        _server = subprocess.Popen([
            "vllm", "serve", MODEL,
            "--host", "127.0.0.1",           # nur die FastAPI-App im selben Container greift zu
            "--port", "8000",
            "--max-model-len", "8192",       # Kontext begrenzen: KV-Cache auf den 24GB der L4 klein halten
            "--gpu-memory-utilization", "0.90",
            "--dtype", "bfloat16",           # explizit: Qwen3.5 kommt in BF16; die L4 unterstützt es
        ])
        # Auf den eigenen billigen /health-Endpoint des Servers warten
        for _ in range(120):
            try:
                requests.get("http://localhost:8000/health", timeout=2).raise_for_status()
                break
            except Exception:
                time.sleep(2)
        else:
            raise RuntimeError("vLLM server did not become ready")
        from transformers import AutoTokenizer
        _tokenizer = AutoTokenizer.from_pretrained(MODEL)
    return _tokenizer

@web_app.get("/health")
def health():
    """Billige Prüfung: den lokalen vLLM-Server fragen, keine Generierung."""
    try:
        requests.get("http://localhost:8000/health", timeout=2).raise_for_status()
        return {"status": "healthy", "model": "Qwen3.5-2B"}
    except Exception:
        # 503, nicht 200: Monitoring soll einen degradierten Service als down sehen
        return JSONResponse(status_code=503,
                            content={"status": "degraded", "model": "Qwen3.5-2B"})

@web_app.post("/chat", response_model=ChatResponse)
def chat(request: ChatRequest, http_request: Request,
         _auth: None = Depends(require_token)):
    """Antwort generieren: Prompt templaten, vLLM die Inferenz überlassen."""
    check_rate_limit(http_request)
    request_id = uuid.uuid4().hex[:12]
    start = time.perf_counter()
    try:
        tokenizer = start_vllm()
        # Instruct-Modelle erwarten ihr Chat-Template, keinen rohen Text.
        # apply_chat_template hüllt die Nachricht in die Special Tokens des Modells.
        messages = [{"role": "user", "content": request.message}]
        # Echtes Token-Budget: den getemplateten Input mit demselben
        # Tokenizer zählen, den vLLM verwendet. Zeichen sind keine Tokens.
        # Bei einer Konversation liefert apply_chat_template einen dict-artigen
        # Batch, dessen "input_ids" die flache Token-Liste ist; len() des
        # Batch selbst würde Keys zählen (2), nicht Tokens.
        input_ids = tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True,
            return_dict=True)["input_ids"]
        if len(input_ids) > MAX_INPUT_TOKENS:
            raise HTTPException(
                422, f"Message too long ({len(input_ids)} tokens; "
                     f"max {MAX_INPUT_TOKENS} input tokens)")
        # Dieselben IDs weiterreichen, die wir gerade gezählt haben:
        # /v1/completions akzeptiert eine Token-ID-Liste als "prompt",
        # sodass vLLM den Text nicht anders re-tokenisieren kann,
        # als wir budgetiert haben.
        r = requests.post(
            "http://localhost:8000/v1/completions",
            json={"model": MODEL, "prompt": input_ids,
                  "max_tokens": request.max_tokens, "temperature": 0.7},
            timeout=120,
        )
        r.raise_for_status()
        body = r.json()
        text = body["choices"][0]["text"].strip()
        latency_ms = (time.perf_counter() - start) * 1000
        log_request(request_id, len(input_ids),
                    body.get("usage", {}).get("completion_tokens", 0),
                    latency_ms, 200)
        return ChatResponse(response=text, latency_ms=round(latency_ms, 2))
    except HTTPException as e:
        log_request(request_id, 0, 0, (time.perf_counter() - start) * 1000, e.status_code)
        raise
    except requests.HTTPError as e:
        # vLLM selbst hat den Request abgelehnt (z.B. Kontext-Overflow):
        # 4xx mit bereinigter Meldung durchreichen, nie den Text des Backends.
        status = e.response.status_code if e.response is not None else 502
        if 400 <= status < 500:
            log_request(request_id, 0, 0, (time.perf_counter() - start) * 1000, status)
            raise HTTPException(status, "The model rejected this request. "
                                        "Try a shorter message.")
        log_request(request_id, 0, 0, (time.perf_counter() - start) * 1000, 502)
        raise HTTPException(502, "Inference backend error. Please try again.")
    except Exception:
        log_request(request_id, 0, 0, (time.perf_counter() - start) * 1000, 500)
        raise HTTPException(500, "Generation failed. Please try again.")

@app.function(
    image=image,
    gpu="L4",
    timeout=600,
    scaledown_window=300,  # 5 Minuten warm halten
    volumes={"/root/.cache/huggingface": model_cache},  # Modell wird einmal heruntergeladen
    secrets=[modal.Secret.from_name("llm-api-token")],  # stellt API_TOKEN bereit
)
@modal.concurrent(max_inputs=10)  # Sicher: der vLLM-Server batcht gleichzeitige Requests
@modal.asgi_app(requires_proxy_auth=True)  # Modal lehnt nicht authentifizierte Requests vor dem Start ab
def serve():
    start_vllm()  # Einmal laden, wenn der Container startet, für jeden Request wiederverwenden
    return web_app

'''

print("Speichere diesen Code als 'llm_service.py':")
print(llm_service_code)


In [ ]:
# Als check_service.py speichern. Das Notebook sendet keine Anfragen.
client_code = '''
import os
import requests

URL = os.environ["LLM_URL"].rstrip("/")  # HTTPS-Basis-URL deines eigenen Deployments
if not URL.startswith("https://"):
    raise ValueError("LLM_URL muss HTTPS verwenden")
proxy = {"Modal-Key": os.environ["MODAL_KEY"], "Modal-Secret": os.environ["MODAL_SECRET"]}
chat = {**proxy, "Authorization": "Bearer " + os.environ["API_TOKEN"]}

# Zuerst den Proxy prüfen: Ohne Header darf keine GPU starten.
r = requests.get(f"{URL}/health", timeout=30, allow_redirects=False)
assert r.status_code == 401, f"Ablehnung durch den Proxy erwartet, erhalten: {r.status_code}"

r = requests.get(f"{URL}/health", headers=proxy, timeout=300, allow_redirects=False)
assert r.status_code == 200 and r.json()["status"] == "healthy", f"Readiness: {r.status_code}"

def ask(headers, payload):
    return requests.post(f"{URL}/chat", headers=headers, json=payload,
                         timeout=150, allow_redirects=False)

r = ask(proxy, {"message": "hi"})  # der Proxy ersetzt die App-Authentifizierung nicht
assert r.status_code == 401, f"Ablehnung durch die App erwartet, erhalten: {r.status_code}"
r = ask({**proxy, "Authorization": "Bearer wrong-token"}, {"message": "hi"})
# Falsches App-Token: HTTP 401 erwartet
assert r.status_code == 401, f"App-Status: {r.status_code}"
r = ask(chat, {"message": " "})
assert r.status_code == 422, f"Validierungsfehler erwartet, erhalten: {r.status_code}"
r = ask(chat, {"message": "Was ist Python?", "max_tokens": 64})
assert r.status_code == 200 and r.json()["response"].strip(), f"Chat: {r.status_code}"
print("Proxy, Bereitschaft, App-Authentifizierung, Validierung und Chat erfolgreich geprüft.")
'''
print(client_code)


### Checkliste für den optionalen Live-Durchlauf im Terminal

1. Lege in der Zielumgebung das Modal-Secret `llm-api-token` mit einem langen, zufälligen `API_TOKEN` an. Es ist weder dein CLI-Zugang noch das Proxy-Token.
2. Speichere `llm_service.py` und `check_service.py`. Richte vor `modal deploy llm_service.py` eine Kostenwarnung ein. Setze `LLM_URL` ausschliesslich auf die HTTPS-Basis-URL aus deinem eigenen Deployment. Lade `MODAL_KEY`, `MODAL_SECRET` und `API_TOKEN` aus dem Secret-Manager in die private Client-Umgebung.
3. Führe `python check_service.py` aus. Erwartet: ohne Header Proxy-401, mit beiden Proxy-Headern `/health` 200. Ein anhaltendes App-503 untersuchst du mit `modal app logs my-llm-service`. Frage sparsam ab: Auch ohne Generierung kann eine autorisierte Probe die GPU aufwecken oder warm halten.
4. Ohne gültigen App-Bearer liefert `/chat` nach Proxy-Freigabe 401, mit gültiger Authentifizierung bei leerer Nachricht 422 und bei gültiger Anfrage eine nichtleere Antwort. Der Client setzt eigene Zeitlimits und folgt mit geheimen Headern keinen Weiterleitungen. `/docs` und `/openapi.json` benötigen ebenfalls die Proxy-Header; `Authorization` bleibt dem App-Token vorbehalten.
5. Auch nach Fehlern: `modal app stop my-llm-service`, im Dashboard auf laufende Container prüfen und nicht mehr benötigte Demo-Zugangsdaten widerrufen.

**Lokaler Abschluss:** Nutze die HTTP-Mocks aus Kapitel 16. Ordne die erwarteten Ablehnungen im ausgegebenen Client dem Proxy bzw. der App zu. Ein lokaler `TestClient` umgeht den Modal-Proxy. Er kann die App prüfen, aber weder die Zugangsprüfung am Proxy noch GPU-Bereitschaft oder Kaltstartkosten bestätigen.


## Abschnitt 17.7: Secrets und Volumes

### Secrets erstellen

```bash
# Ein Secret über die Kommandozeile erstellen
modal secret create my-secrets API_KEY=your_secret_key
```

### Secrets im Code verwenden

In [ ]:
# Secrets in deinem Code verwenden
secrets_example = '''
@app.function(secrets=[modal.Secret.from_name("my-secrets")])
def with_secrets():
    import os
    api_key = os.environ["API_KEY"]
    # Den Schlüssel sicher verwenden...
'''

print("Secrets verwenden:")
print(secrets_example)

In [ ]:
# Volumes zum Cachen von Modellen verwenden
volumes_example = '''
# Ein Volume für Modell-Cache erstellen
model_cache = modal.Volume.from_name("model-cache", create_if_missing=True)

@app.function(
    gpu="L4",
    volumes={"/root/.cache/huggingface": model_cache},
)
def with_cache():
    # Modelle werden einmal heruntergeladen, dann im Volume gecacht
    pass
'''

print("Volumes zum Cachen von Modellen verwenden:")
print(volumes_example)
print("\nErste Anfrage lädt das Modell herunter. Nachfolgende Anfragen verwenden den Cache.")

## Abschnitt 17.8: GPU-Optionen

| Modellgrösse | Empfohlene GPU | Kosten/Stunde |
|------------|-----------------|----------|
| < 3B Parameter (FP16/quantisiert) | T4 (16GB) | 0,59$ |
| < 3B Parameter (BF16, z.B. Qwen3.5-2B) | L4 (24GB) | 0,80$ |
| 3-8B Parameter | A10 (24GB) | 1,10$ |
| 8-30B Parameter | A100-40GB | 2,50$ |
| 30B+ Parameter | H100 | 3,95 $ |

(Preise geprüft 2026-11; verifiziere auf modal.com/pricing.)

**Beginne mit L4.** Die Gewichte von Qwen3.5-2B sind BF16, und vLLM 0.34 verweigert BF16 auf GPUs älter als Ampere (Compute Capability 8.0): Die T4 hat 7.5 und würde beim Start abbrechen, L4 (8.9) und A10 (8.6) schaffen die Hürde. Die T4 funktioniert weiter für kleinere FP16- oder quantisierte Modelle. Upgrade nur bei Bedarf.

In [ ]:
# GPU-Auswahl-Beispiele
print("GPU-Auswahl ist ein Parameter:")
print()
print('@app.function(gpu="T4")      # 16GB, 0,59$/Std - nur FP16- oder quantisierte Modelle (kein BF16)')
print('@app.function(gpu="L4")      # 24GB, 0,80$/Std - unsere Wahl für Qwen3.5-2B')
print('@app.function(gpu="A10")     # 24GB, 1,10$/Std - gleicher Speicher, schnellere Berechnung')
print('@app.function(gpu="A100")    # Hohe Leistung')
print('@app.function(gpu="H100")    # Maximale Leistung')
print('@app.function(gpu="A100:2")  # Zwei A100s')

## Abschnitt 17.9: Dein Deployment verwalten

### Logs anzeigen

```bash
modal app logs my-llm-service
```

Oder verwende das Dashboard unter [modal.com](https://modal.com).

### Deine App aktualisieren

```bash
# Einfach neu deployen
modal deploy llm_service.py
```

Keine Ausfallzeit. Modal übernimmt automatisch Rolling Updates.

### Kostenkontrolle

- Verwende `scaledown_window`, um inaktive Container herunterzufahren
- Beginne mit L4, upgrade nur bei Bedarf
- Richte Ausgaben-Alarme im Modal-Dashboard ein; `modal app stop my-llm-service` baut alles ab
- Überwache Ausgaben im Modal-Dashboard

Das Funktionslimit von 600 Sekunden ersetzt kein HTTP-Timeout. Die Grenzen von 4.000 Zeichen, 2.048 Tokens am formatierten Input und 512 Output-Tokens gelten für eine einzelne Anfrage; weder sie noch Kostenwarnungen begrenzen die Gesamtausgaben. Der IP-Zähler (20 Anfragen pro Minute) gilt je Container und wird beim Herunterfahren zurückgesetzt. Für mehrere Benutzer und Replikate brauchst du benutzerbezogene Kontingente in einem gemeinsam genutzten Speicher mit atomaren Aktualisierungen.

## Zusammenfassung

Du hast gelernt, wie man:

1. **Funktionen zu Modal deployed** mit `@app.function()`
2. **GPU-Unterstützung hinzufügt** mit einem einzigen Parameter: `gpu="L4"`
3. **Abhängigkeiten installiert** mit `modal.Image`
4. **Alle Routen vor dem Containerstart schützt** mit `@modal.asgi_app(requires_proxy_auth=True)`
5. **Secrets sicher speichert** mit `modal.Secret`
6. **Modelle cacht** mit `modal.Volume`
7. **Kosten kontrolliert** mit Idle-Timeouts und Monitoring

**Teile die Oberfläche, nicht die Zugangsdaten.** Die URL allein berechtigt zu nichts. Das Workspace-Proxy-Paar bleibt in einem vertrauenswürdigen Client oder auf deinem Server; für `/chat` braucht dieser zusätzlich das getrennte App-Token. Für Endnutzer schaltest du eine eigene authentifizierte Oberfläche davor. Das Notebook allein belegt kein Live-Deployment.

**Befehle zum Merken:**
```bash
pip install modal      # Installieren
modal setup            # Authentifizieren
modal run file.py      # Einmal ausführen
modal deploy file.py   # Bis zum Stoppen bereitstellen
modal app logs name    # Logs anzeigen
modal app stop name    # Dienst stoppen
```